# Lab 15: Diffusion Models (DDPM) & Generative Image Synthesis

Welcome to Laboratory 15! In this lab, we implement the core mechanics of **Denoising Diffusion Probabilistic Models (DDPM)** (*Ho et al., 2020*):
1. **The Forward Diffusion Process ($q$)**: Incrementally perturb clean images with Gaussian noise across $T$ discrete timesteps.
2. **Closed-Form Noising Trick**: Sample noisy image $x_t$ directly from $x_0$ at any arbitrary timestep $t$ using cumulative variance product $\bar{\alpha}_t$.
3. **Reverse Denoising Architecture ($p_\theta$)**: Formulate the noise prediction objective $\min_\theta \|\boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)\|^2$.


## 1. Technical Preliminaries & Imports


In [ ]:
# Import PyTorch and numerical libraries for diffusion modeling
import torch
import torch.nn as nn
import numpy as np

# Seed for reproducibility
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Active Compute Device:', device)


## 2. DDPM Forward Variance Schedule & Closed-Form Sampling

### Mathematical Formulation
Given clean sample $\mathbf{x}_0 \sim q(\mathbf{x}_0)$ and variance schedule $\beta_1, \dots, \beta_T$:
* $\alpha_t = 1 - \beta_t$
* $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$ (Cumulative product of variances)

### Closed-Form Direct Sampling Formula:
$$\mathbf{x}_t = \sqrt{\bar{\alpha}_t} \mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t} \boldsymbol{\epsilon}, \quad \text{where } \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

This formulation allows jump-sampling to any timestep $t$ in $\mathcal{O}(1)$ without iterating through $1, 2, \dots, t-1$!


### Helper Class: `DiffusionSchedule`
The `DiffusionSchedule` class computes $\beta_t$, $\alpha_t$, $\bar{\alpha}_t$ and provides the closed-form `q_sample` forward noising method.


In [ ]:
# Define the DDPM Forward Diffusion Variance Schedule
class DiffusionSchedule:
    """Computes DDPM forward variance schedules and performs closed-form noising."""
    def __init__(self, timesteps: int = 200, beta_start: float = 0.0001, beta_end: float = 0.02):
        self.timesteps = timesteps
        
        # 1. Linear variance schedule from beta_start to beta_end
        self.betas = torch.linspace(beta_start, beta_end, timesteps)
        
        # 2. Alpha schedule: alpha_t = 1 - beta_t
        self.alphas = 1.0 - self.betas
        
        # 3. Alpha cumulative products: alpha_bar_t = prod_{s=1}^t alpha_s
        self.alpha_hat = torch.cumprod(self.alphas, dim=0)
        
    def q_sample(self, x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor = None) -> tuple:
        """Samples noisy image x_t directly from clean image x0 at timestep t.
        
        Args:
            x0: Clean image tensor of shape (Batch_Size, Channels, Height, Width)
            t: Timestep indices tensor of shape (Batch_Size,)
            noise: Optional Gaussian noise tensor. If None, samples from N(0, I).
        Returns:
            x_t: Noisy image tensor at timestep t.
            noise: The ground-truth noise epsilon that was injected.
        """
        if noise is None:
            noise = torch.randn_like(x0) # Sample epsilon ~ N(0, I)
            
        # Extract sqrt(alpha_bar_t) and sqrt(1 - alpha_bar_t) and reshape for broadcasting: (B, 1, 1, 1)
        sqrt_alpha_bar = torch.sqrt(self.alpha_hat[t]).view(-1, 1, 1, 1).to(x0.device)
        sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - self.alpha_hat[t]).view(-1, 1, 1, 1).to(x0.device)
        
        # Closed-form forward noising equation: x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * epsilon
        x_t = sqrt_alpha_bar * x0 + sqrt_one_minus_alpha_bar * noise
        return x_t, noise

# Instantiate schedule with T=200 timesteps
diffusion_sched = DiffusionSchedule(timesteps=200)

# Simulate noising an image tensor
clean_image = torch.zeros((1, 1, 28, 28)) # Clean image x_0
target_timestep = torch.tensor([100])      # Midpoint timestep t=100

noisy_image, injected_noise = diffusion_sched.q_sample(clean_image, target_timestep)

print(f'Clean Image Shape:    {clean_image.shape}')
print(f'Sampled x_{target_timestep.item()} Shape: {noisy_image.shape}')
print(f'Alpha_bar at t=100:   {diffusion_sched.alpha_hat[100].item():.4f}')
print('[Verification Passed] Closed-form DDPM forward sampling succeeded!')


## 3. Summary & Key Takeaways
1. **Forward Process ($q$)**: Converts structured data distributions into pure isotropic Gaussian noise through a fixed Markov chain.
2. **Reparameterization**: Cumulative variance $\bar{\alpha}_t$ allows direct closed-form sampling of any intermediate state $x_t$ during training.
3. **Generative Sampling**: Image generation inverts the forward process by training a neural network to predict and subtract noise at each timestep.
